# 3c — Similarity Sweeps (v2.0)

Kernel bandwidth (σ) and K optimisation for the **Similarity model**.

| § | Stage | Description | Est. wall time |
|---|---|---|---|
| 2 | **Sanity check** | σ=1.0, K=7 baseline | 1–5 min |
| 3 | **σ Optuna** | Per-feature bandwidth TPE | 2–4 h |
| 4 | **K sweep** | Grid over K with best σ | 10–30 min |
| 5 | **Feature sensitivity** | LOO + single-feature + σ heatmap | 1–3 h |

Outputs → `output/sweeps/sim_*.csv` | Figures → `fig/sweeps/sim_*.png`

Best params written to `output/sweeps/sim_best_params.json` (imported by `4_MODEL.ipynb`).

**Kernel:** \( w(x_t, x_r) = \exp\!\left(-\tfrac{1}{2} \overline{\left(\frac{x_t - x_r}{\sigma_f}\right)^2}\right) \)

σ is optimised jointly via Optuna TPE with K held at `K_FIXED_FOR_SIGMA` = 7,  then K is swept over `KRANGE` with the best σ fixed.

## 1. Imports & setup

In [1]:
import sys, json, time, warnings, platform, psutil, datetime
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from config import *

optuna.logging.set_verbosity(optuna.logging.WARNING)

sweep_dir.mkdir(parents=True, exist_ok=True)
(fig_root / 'sweeps').mkdir(parents=True, exist_ok=True)
fig_sweep = fig_root / 'sweeps'

print(f"sweep_dir : {sweep_dir}")
print(f"fig_sweep : {fig_sweep}")


✓ config.py v4.0 | obs_model: 21 | obs_sweep: 22 | obs: 36
sweep_dir : output/sweeps
fig_sweep : fig/sweeps


## 0. Toggles

In [2]:
RUN_SANITY      = True
RUN_SIGMA_SWEEP = True
RUN_K_SWEEP     = True
RUN_FEATURE_SENS= True
RUN_LOO         = True
RUN_SINGLE      = True
RUN_SIGMA_SENS  = True

K_FIXED_FOR_SIGMA  = 7.0
K_SWEEP_STEPS      = 25
SIGMA_SENS_STEPS   = 15
N_OPTUNA_SIM       = N_OPTUNA_TRIALS


### 1a. Verify parquet

In [3]:
df_check = pd.read_parquet(parquet_ref, columns=obs_model + ['q'])
missing_m = [c for c in obs_model if c not in df_check.columns]
all_nan_m = [c for c in obs_model if df_check[c].isna().all()]
assert not missing_m, f'MISSING from parquet: {missing_m}'
assert not all_nan_m, f'ALL-NaN columns: {all_nan_m}'
print(f'\u2713 {len(obs_model)} obs_model features verified | q_clip_max={q_clip_max}')


✓ 21 obs_model features verified | q_clip_max=0.35


### 1b. Load reference data

In [4]:
obs_sel = obs_sweep  # NOTE: adjust here to test a different feature universe

df_ref    = pd.read_parquet(parquet_ref)
available = [f for f in obs_sel if f in df_ref.columns]
missing   = [f for f in obs_sel if f not in df_ref.columns]
if missing:
    warnings.warn(f'obs_sel columns missing from parquet: {missing}')
    raise RuntimeError(f'Missing features: {missing}')

cols_needed = available + ['q', 'lat', 'lon']
if 'weight' in df_ref.columns:
    cols_needed.append('weight')

work = (df_ref[cols_needed]
        .replace([np.inf, -np.inf], np.nan)
        .dropna())
work = work[(work['q'] >= q_clip_min) & (work['q'] <= q_clip_max)]

y_phys = work['q'].values
w_all  = work['weight'].values if 'weight' in work.columns else np.ones(len(work))
X_all  = work[available].values

print(f'Reference rows : {len(work):,}')
print(f'Features       : {len(available)}')
print(available)


Reference rows : 30,847
Features       : 22
['MOHO', 'LITH_RHO', 'CTD', 'EMAG2_LOG', 'SI', 'CRUST_RHO', 'REVEAL_VP60VS70', 'LAB', 'MAG_SEIS_MOHO', 'GEOID', 'DEM', 'BOUGUER', 'MOHO_GRAV', 'REVEAL_P150', 'REVEAL_VP90VS60', 'REVEAL_S80', 'SEDIMENT', 'REVEAL_VP50VS80', 'REVEAL_S100', 'REVEAL_S70', 'FREE_AIR', 'REVEAL_S90']


### 1c. Build CV folds + similarity helper

In [ ]:
from lib.similarity import (
    prepare_sigma, standardise_features,
    similarity_sweep, similarity_predict,
    spatial_block_kfold
)

df_sim = (df_ref[obs_sel + ['q', 'lat', 'lon']]
           .replace([np.inf, -np.inf], np.nan)
           .dropna())
df_sim = df_sim[(df_sim['q'] >= q_clip_min) & (df_sim['q'] <= q_clip_max)]
X_sim  = df_sim[obs_sel].values.astype(np.float32)
y_sim  = df_sim['q'].values.astype(np.float32)
w_sim  = df_sim['weight'].values.astype(np.float32) if 'weight' in df_sim.columns else np.ones(len(df_sim), dtype=np.float32)

kf_sim      = KFold(n_splits=CV_FOLDS_SIM, shuffle=True, random_state=random_state)
fold_splits = list(kf_sim.split(X_sim))

# fold_splits = spatial_block_kfold(
#     df_sim['lat'].values, df_sim['lon'].values,
#     n_splits=CV_FOLDS_SIM, random_state=random_state,
# )

print(f'Sim reference rows : {len(df_sim):,}')
print(f'Sim features       : {len(obs_sel)}')

SIM_KERNEL = "rbf"
SIM_WEIGHT_MODE = "shifted_exp"

print(f"Similarity mode: kernel={SIM_KERNEL} | weight_mode={SIM_WEIGHT_MODE}")

def sim_cv_r2(X_raw, y, sigma_arr, K, fold_splits, w=None):
    """
    CV R² using similarity_predict (numba JIT — no chunking needed).
    StandardScaler fitted on train fold only; sigma_arr applied after scaling.
    Uses the updated robust similarity defaults explicitly for reproducibility.
    """
    sigma_f32 = np.asarray(sigma_arr, dtype=np.float32)
    r2_folds = []
    for tr, te in fold_splits:
        sc_cv = StandardScaler().fit(X_raw[tr])
        X_trn = sc_cv.transform(X_raw[tr]).astype(np.float32)
        X_ten = sc_cv.transform(X_raw[te]).astype(np.float32)
        w_tr = w[tr].astype(np.float32) if w is not None else np.ones(len(tr), dtype=np.float32)
        w_te = np.ones(len(te), dtype=np.float32)
        y_tr = y[tr].astype(np.float32)

        q_pred, _, _, _ = similarity_predict(
            X_ref=X_trn,
            H_ref=y_tr,
            X_tgt=X_ten,
            W_t=w_te,
            S_r=sigma_f32,
            K=float(K),
            W_r=w_tr,
            kernel=SIM_KERNEL,
            weight_mode=SIM_WEIGHT_MODE,
            use_sigma_helper=False,
        )
        q_pred = np.clip(q_pred, q_clip_min, q_clip_max)
        valid = np.isfinite(q_pred)
        if valid.sum() < 10:
            return -1.0
        r2_folds.append(r2_score(y[te][valid], q_pred[valid]))
    return float(np.mean(r2_folds))


SIGMA_START     = {f: 1.0 for f in obs_sel}
sigma_start_arr = np.ones(len(obs_sel), dtype=np.float32)


Sim reference rows : 30,847
Sim features       : 22
Similarity mode: kernel=rbf | weight_mode=shifted_exp


---
## 2. Sanity check (σ=1.0, K=7)

Quick verification that the CV loop runs correctly before expensive optimisation.

In [ ]:
r2_baseline_sim = sim_cv_r2(X_sim, y_sim, sigma_start_arr, K_FIXED_FOR_SIGMA, fold_splits)
print(f'Baseline CV R²  (σ=1.0, K={K_FIXED_FOR_SIGMA}) = {r2_baseline_sim:.4f}')
if r2_baseline_sim < -1:
    print('⚠ R² strongly negative — check data / parquet before sweeping.')
elif r2_baseline_sim < 0:
    print('R² slightly negative — σ=1.0 too tight; Optuna should improve.')
else:
    print('Positive baseline — proceeding to optimisation.')


In [7]:
from sklearn.preprocessing import StandardScaler
from lib.similarity import sim_gaussian_2d

tr0, te0 = fold_splits[0]
sc_tmp = StandardScaler().fit(X_sim[tr0])
X_tr0 = sc_tmp.transform(X_sim[tr0]).astype(np.float32)
X_te0 = sc_tmp.transform(X_sim[te0[:20]]).astype(np.float32)

S_diag = sim_gaussian_2d(X_tr0[:500], X_te0, sigma_start_arr)
print(f"sigma_start_arr: min={sigma_start_arr.min():.3f}  max={sigma_start_arr.max():.3f}")
print(f"S range:  {S_diag.min():.3e}  –  {S_diag.max():.3e}")
print(f"S > 0.01: {(S_diag > 0.01).mean():.4f}")
print(f"S > 1e-5: {(S_diag > 1e-5).mean():.4f}")
print(f"n_feat={X_sim.shape[1]},  n_tr={len(tr0)},  n_te={len(te0)}")
print(f"y_sim range: {y_sim.min():.4f} – {y_sim.max():.4f}")
print(f"q_clip_min={q_clip_min},  q_clip_max={q_clip_max}")

sigma_start_arr: min=1.000  max=1.000
S range:  0.000e+00  –  4.424e-01
S > 0.01: 0.0603
S > 1e-5: 0.3361
n_feat=22,  n_tr=24770,  n_te=6077
y_sim range: 0.0010 – 0.3500
q_clip_min=0.001,  q_clip_max=0.35


In [10]:
# Need lat/lon in df_sim for this
if 'lat' in df_sim.columns and 'lon' in df_sim.columns:
    for i, (tr, te) in enumerate(fold_splits):
        lat_te = df_sim['lat'].values[te]
        lon_te = df_sim['lon'].values[te]
        print(f"Fold {i}: lat [{lat_te.min():.1f}, {lat_te.max():.1f}]  "
              f"lon [{lon_te.min():.1f}, {lon_te.max():.1f}]  n={len(te)}")

Fold 0: lat [-64.6, 57.2]  lon [-180.0, 179.8]  n=6077
Fold 1: lat [-69.0, 86.2]  lon [-177.4, 35.9]  n=15893
Fold 2: lat [-64.1, 89.0]  lon [-107.8, 179.9]  n=2748
Fold 3: lat [-38.0, 88.4]  lon [-179.9, 107.9]  n=4140
Fold 4: lat [-70.0, 25.4]  lon [-177.3, 179.9]  n=1989


---
## 3. σ Optimisation (Optuna)

Joint optimisation of per-feature σ with K fixed at `K_FIXED_FOR_SIGMA`.  
σ bounds: `SIGMA_BOUNDS` = (0.25, 4.0) log-scale.  
Warm-started with σ=1.0 for all features.

In [ ]:
sigma_csv      = sweep_dir / 'sim_sigma_results.csv'
t_sig_start    = time.time()

if RUN_SIGMA_SWEEP:
    def _objective_sigma(trial):
        s_arr = np.array([
            trial.suggest_float(f, SIGMA_BOUNDS[0], SIGMA_BOUNDS[1], log=True)
            for f in obs_sel
        ])
        return sim_cv_r2(X_sim, y_sim, s_arr, K_FIXED_FOR_SIGMA, fold_splits)

    study_sigma = optuna.create_study(direction='maximize',
                                      sampler=optuna.samplers.TPESampler(seed=random_state))
    study_sigma.enqueue_trial({f: 1.0 for f in obs_sel})
    study_sigma.optimize(_objective_sigma, n_trials=N_OPTUNA_SIM, show_progress_bar=True)
    best_sigmas   = {f: study_sigma.best_params[f] for f in obs_sel}
    best_sigma_r2 = study_sigma.best_value
    study_sigma.trials_dataframe().to_csv(sigma_csv, index=False)
    print(f'\nCV R² (opt σ, K={K_FIXED_FOR_SIGMA}) = {best_sigma_r2:.4f}')
    for f, v in best_sigmas.items(): print(f'  {f:28s} {v:.4f}')
elif sigma_csv.exists():
    df_trials     = pd.read_csv(sigma_csv)
    best_row_s    = df_trials.loc[df_trials['value'].idxmax()]
    best_sigmas   = {f: float(best_row_s[f'params_{f}']) for f in obs_sel}
    best_sigma_r2 = float(best_row_s['value'])
    print(f'Loaded {sigma_csv}  ({len(df_trials)} trials)  best R²={best_sigma_r2:.4f}')
else:
    print('No saved results — set RUN_SIGMA_SWEEP = True.')

best_sigmas_arr = np.array([best_sigmas[f] for f in obs_sel])
t_sig_end = time.time()

# approx 1.10 min each iteration


---
## 4. K Sweep

Grid search over K with best σ fixed. `KRANGE` = (2.0, 20.0), `K_SWEEP_STEPS` = 25.

In [ ]:
# Diagnose N_sim scale
sc_tmp = StandardScaler().fit(X_sim)
X_tmp = sc_tmp.transform(X_sim[:100]).astype(np.float32)
X_ref_tmp = sc_tmp.transform(X_sim[:500]).astype(np.float32)
from lib.similarity import sim_gaussian_2d
S = sim_gaussian_2d(X_ref_tmp, X_tmp, best_sigmas_arr)
print(f"N_sim (raw sum) range: {S.sum(axis=1).min():.1f} – {S.sum(axis=1).max():.1f}")
print(f"K^N_sim at K=2: {2**S.sum(axis=1).mean():.3e}")  # if inf, confirmed

In [ ]:
k_csv      = sweep_dir / 'sim_k_results.csv'
t_k_start  = time.time()

if RUN_K_SWEEP:
    k_vals = np.linspace(K_RANGE[0], K_RANGE[1], K_SWEEP_STEPS)
    k_rows = []
    for kv in k_vals:
        r2 = sim_cv_r2(X_sim, y_sim, best_sigmas_arr, kv, fold_splits)
        k_rows.append(dict(K=kv, cv_r2=r2))
        print(f'  K={kv:5.2f}  CV R²={r2:.4f}')
    k_df      = pd.DataFrame(k_rows)
    k_df.to_csv(k_csv, index=False)
    best_K    = float(k_df.loc[k_df['cv_r2'].idxmax(), 'K'])
    best_K_r2 = float(k_df['cv_r2'].max())
    print(f'\nBest K = {best_K:.2f}  CV R² = {best_K_r2:.4f}')
    print(f'Saved {k_csv}')
elif k_csv.exists():
    k_df      = pd.read_csv(k_csv)
    best_K    = float(k_df.loc[k_df['cv_r2'].idxmax(), 'K'])
    best_K_r2 = float(k_df['cv_r2'].max())
    print(f'Loaded {k_csv}  best K={best_K:.2f}  R²={best_K_r2:.4f}')
else:
    best_K, best_K_r2, k_df = K_FIXED_FOR_SIGMA, None, None
    print('No saved results — set RUN_K_SWEEP = True.')

t_k_end = time.time()


### 4a. K sweep figure

In [ ]:
if k_df is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(k_df['K'], k_df['cv_r2'], 'o-', ms=5, lw=1.5, color='steelblue')
    ax.axvline(best_K, color='firebrick', lw=1.5, ls='--', label=f'Best K={best_K:.2f}')
    ax.set_xlabel('K', fontsize=10);  ax.set_ylabel('CV R²', fontsize=10)
    ax.set_title('Similarity K sweep', fontsize=11)
    ax.legend(fontsize=9);  ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(fig_sweep/'sim_k_sweep.png', dpi=FIG_DPI, bbox_inches='tight')
    plt.show();  print(f'Saved {fig_sweep}/sim_k_sweep.png')


---
## 5. Feature Sensitivity

**5a LOO** — drop one feature at a time with best σ and K.  
**5b Single** — univariate R² for each feature.  
**5c σ heatmap** — vary one σ, hold others at best.

In [ ]:
sim_loo_csv    = sweep_dir / 'sim_loo_results.csv'
sim_single_csv = sweep_dir / 'sim_single_results.csv'
sim_sens_csv   = sweep_dir / 'sim_sigma_sensitivity.csv'
t_sens_start   = time.time()

r2_full = sim_cv_r2(X_sim, y_sim, best_sigmas_arr, best_K, fold_splits)
print(f'Full model CV R² (best σ, best K={best_K:.2f}) = {r2_full:.4f}')

# ── LOO ───────────────────────────────────────────────────────
if RUN_FEATURE_SENS and RUN_LOO:
    loo_rows = []
    for i, feat in enumerate(obs_sel):
        mask_i = np.ones(len(obs_sel), dtype=bool);  mask_i[i] = False
        r2_d   = sim_cv_r2(X_sim[:, mask_i], y_sim, best_sigmas_arr[mask_i], best_K, fold_splits)
        delta  = r2_d - r2_full
        tag    = 'HELPFUL' if delta < -0.002 else ('HARMFUL' if delta > 0.002 else 'neutral')
        loo_rows.append(dict(feature=feat, r2_without=r2_d, delta_r2=delta, verdict=tag))
        print(f'{i+1:3d}/{len(obs_sel)} drop {feat:28s}  R²={r2_d:.4f}  Δ={delta:+.4f}  {tag}')
    loo_df = pd.DataFrame(loo_rows).sort_values('delta_r2')
    loo_df.to_csv(sim_loo_csv, index=False)
    print(f'Saved {sim_loo_csv}')
elif sim_loo_csv.exists():
    loo_df = pd.read_csv(sim_loo_csv);  print(f'Loaded {sim_loo_csv}')
else:
    loo_df = None

# ── Single-feature R² ──────────────────────────────────────────
if RUN_FEATURE_SENS and RUN_SINGLE:
    single_rows = []
    for i, feat in enumerate(obs_sel):
        r2_s = sim_cv_r2(X_sim[:, [i]], y_sim, np.array([best_sigmas_arr[i]]), best_K, fold_splits)
        single_rows.append(dict(feature=feat, r2_univariate=r2_s))
        print(f'  {feat:28s}  single R²={r2_s:.4f}')
    single_df = pd.DataFrame(single_rows)
    single_df.to_csv(sim_single_csv, index=False)
    print(f'Saved {sim_single_csv}')
elif sim_single_csv.exists():
    single_df = pd.read_csv(sim_single_csv);  print(f'Loaded {sim_single_csv}')
else:
    single_df = None

# ── Per-feature σ sensitivity ─────────────────────────────────
if RUN_FEATURE_SENS and RUN_SIGMA_SENS:
    sens_rows = []
    for i, feat in enumerate(obs_sel):
        s_arr_m = best_sigmas_arr.copy()
        for sv in np.linspace(SIGMA_BOUNDS[0], SIGMA_BOUNDS[1], SIGMA_SENS_STEPS):
            s_arr_m[i] = sv
            sens_rows.append(dict(feature=feat, sigma=sv,
                cv_r2=sim_cv_r2(X_sim, y_sim, s_arr_m, best_K, fold_splits)))
        s_arr_m[i] = best_sigmas_arr[i]
    sigma_sens_df = pd.DataFrame(sens_rows)
    sigma_sens_df.to_csv(sim_sens_csv, index=False)
    print(f'Saved {sim_sens_csv}')
elif sim_sens_csv.exists():
    sigma_sens_df = pd.read_csv(sim_sens_csv);  print(f'Loaded {sim_sens_csv}')
else:
    sigma_sens_df = None

t_sens_end = time.time()


### 5a. Sensitivity figures

In [ ]:
if 'loo_df' in dir() and loo_df is not None and 'single_df' in dir() and single_df is not None:
    rank_df = (loo_df[['feature','r2_without','delta_r2','verdict']]
               .merge(single_df[['feature','r2_univariate']], on='feature'))
    if 'sigma_sens_df' in dir() and sigma_sens_df is not None:
        sens_best = (sigma_sens_df.sort_values('cv_r2', ascending=False)
                     .groupby('feature').first()[['sigma']]
                     .rename(columns={'sigma':'sigma_marginal_best'})
                     .reset_index())
        rank_df = rank_df.merge(sens_best, on='feature', how='left')
    rank_df = rank_df.merge(
        pd.DataFrame({'feature': obs_sel, 'sigma_joint_opt': best_sigmas_arr}),
        on='feature', how='left').round(4)
    rank_csv = sweep_dir / 'sim_feature_ranking.csv'
    rank_df.to_csv(rank_csv, index=False)
    print(f'Full model R² = {r2_full:.4f}')
    print(rank_df.to_string(index=False))
    print(f'Saved {rank_csv}')

if 'sigma_sens_df' in dir() and sigma_sens_df is not None:
    pivot = sigma_sens_df.pivot_table(index='feature', columns='sigma', values='cv_r2')
    feat_order = pivot.max(axis=1).sort_values(ascending=False).index
    pivot = pivot.loc[feat_order]
    fig, ax = plt.subplots(figsize=(12, max(5, len(feat_order)*0.38)))
    im = ax.imshow(pivot.values, aspect='auto',
                   vmin=max(pivot.values.min(), r2_full-0.05), vmax=r2_full+0.02)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{v:.2f}' for v in pivot.columns], fontsize=7, rotation=45)
    ax.set_yticks(range(len(feat_order)));  ax.set_yticklabels(feat_order, fontsize=8)
    ax.set_xlabel('σ (others fixed at best)', fontsize=9)
    ax.set_title(f'Sim per-feature σ sensitivity  R²={r2_full:.4f}  K={best_K:.2f}', fontsize=11)
    plt.colorbar(im, ax=ax, label='CV R²', shrink=0.6)
    fig.tight_layout()
    fig.savefig(fig_sweep/'sim_sigma_sensitivity.png', dpi=FIG_DPI, bbox_inches='tight')
    plt.show();  print(f'Saved {fig_sweep}/sim_sigma_sensitivity.png')


---
## 6. Write `sim_best_params.json`

In [ ]:
sim_json = param_paths['sim']

sim_out = dict(
    model='Similarity', sweep_notebook='3c_SIM_SWEEP',
    SIM_BEST_K      = float(best_K),
    SIM_SIGMAS      = {f: float(best_sigmas[f]) for f in obs_sel},
    kernel          = 'exp(-0.5 * mean((xt-xr / sigma_f)^2))',
    standardisation = 'StandardScaler per CV fold (train fold only)',
    features        = obs_sel,
    obs_model       = obs_model,
    cv_r2_baseline  = float(r2_baseline_sim),
    cv_r2_sigma_opt = float(best_sigma_r2) if best_sigma_r2 is not None else None,
    cv_r2_k_opt     = float(best_K_r2)     if best_K_r2     is not None else None,
    cv_r2_full      = float(r2_full),
    K_fixed_sigma   = K_FIXED_FOR_SIGMA,
    n_optuna_trials = N_OPTUNA_SIM,
)
with open(sim_json, 'w') as fp: json.dump(sim_out, fp, indent=2)
print(f'Saved {sim_json}')
print(f'Best K     : {best_K:.2f}')
print(f'CV R² full : {r2_full:.4f}')


# Saved output/sweeps/sim_best_params.json
# Best K     : 6.50
# CV R² full : 0.4409

# Saved output/sweeps/sim_best_params.json
# Best K     : 7.25
# CV R² full : 0.4760

---
## 7. Runtime log

In [ ]:
def _safe(x):
    try: return float(x)
    except: return None

log_df = pd.DataFrame([
    dict(stage='sim_sigma_optuna', wall_s=_safe(locals().get('t_sig_end',0)   - locals().get('t_sig_start',0))),
    dict(stage='sim_k_sweep',      wall_s=_safe(locals().get('t_k_end',0)     - locals().get('t_k_start',0))),
    dict(stage='sim_feature_sens', wall_s=_safe(locals().get('t_sens_end',0)  - locals().get('t_sens_start',0))),
])
log_df['wall_min'] = (log_df['wall_s'] / 60).round(2)
log_df.to_csv(sweep_dir/'sim_runtime_log.csv', index=False)
print(log_df[['stage','wall_min']].to_string(index=False))
print(f'Saved {sweep_dir}/sim_runtime_log.csv')
